发送给大模型的所有消息都可以称为提示词（Prompt），它直接影响模型的输出结果。
其中，SystemMessage尤为重要，我们把SystemMessage称为系统提示词（System Prompt），它可以给模型设定角色和本次聊天的背景，对模型生成的内容有很大的影响。

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()
agent = create_agent(
    model="deepseek-chat",
    system_prompt="你以海盗口吻来回答用户问题"
)

for token,metada in agent.stream(
        {"messages":[HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

嘿！小子，站稳了！我是这片海域上最臭名昭著的海盗船长——黑帆的杰克！我的船“咆哮海妖号”可是能让商船闻风丧胆的存在！你问我是谁？哈哈，我可是七海之上自由自在的掠夺者，海风的宠儿，宝藏的追寻者！不过，别想着打听我的藏宝图，那可是我的秘密！说，你找我有什么事？

通过优化System Prompt从而让模型输出更理想的结果的这一过程，我们称为提示词工程（Prompt Engineering）。

也就是说，提示词优化不是一锤子买卖，而是一个不断优化、测试、再优化的过程。那么，提示词到底该怎么写呢？

从内容来说，提示词通常包含以下几个部分，通常按此顺序排列：
- 身份（Identity）：描述AI的职责、沟通风格和总体目标。
- 说明（Instructions）：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- 示例（Examples）：提供可能的输入示例，以及模型期望的输出。
- 背景信息（Context）：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。

从格式来说，在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。
- Markdown 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- XML 标签可以帮助明确区分一段内容（例如用作参考的辅助文档、对话示例等）的起始和结束位置。

In [5]:
system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

school_name = "黑马程序员"

有的时候我们希望模型按照固定的风格来回答问题，而这种风格又不太好描述，那我们就可以通过举例的方式让模型学习例子来回答。
用户只需在输入提示（Prompt）中提供几个输入-输出示例，模型就能理解任务模式并生成符合预期的输出：

In [6]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

云浮城（Aeropolis）—— 悬浮于金星大气层约55公里高空的浮空都市，由一系列巨大的反重力云筏平台串联而成。这里是金星唯一人类可宜居的高度带，气压与温度接近地球，城市以采集大气中的硫酸云滴提炼水和工业原料为生，外围环绕着缓缓旋转的防护云环，用以抵御金星高空狂暴的酸性风暴。

由于传统程序识别结构化的数据会更加方便，所以有时候我们希望LLM也能输出固定结构的内容，方便我们解析。这同样可以通过系统提示词来实现。

In [7]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response['messages'][-1].content)

{
    "name": "晨星城（Phosphoros）",
    "location": "位于金星上层大气55-60公里高度，漂浮于硫酸云层之上的巨大浮空平台群",
    "vibe": "炽热、坚韧、逐日",
    "economy": "硫酸云资源提取、高温材料制造、大气浮空观光、太阳能中继站"
}


In [8]:
from pydantic import BaseModel
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [12]:
# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
city = response['structured_response']
print(city)

name='静海城' location='月球北半球静海（Mare Tranquillitatis）西缘，阿波罗11号着陆区以南约80公里，依托熔岩管群与月海玄武岩台地建造，地下城体深达200米，地表以穹顶港和升降井与轨道电梯相连' vibe='冷寂与繁华并存。地表是永恒的黑白天穹与低重力尘暴，地下是永不熄灭的霓虹熔岩廊道；居民被称为"潮汐人"，说话慢、走路轻，习惯以敲击管壁代替敲门。整座城按地球时运行，却把"月昼"当作节日——每逢月昼开端，全城穹顶会同步降下人造暮色，人们称之为"长夜礼"' economy='以氦-3开采与聚变燃料精炼为支柱，垄断地月能源贸易；水冰冶炼与氧氮分离支撑生命维持系统出口；月球轨道电梯使静海城成为地月物流中枢，同时发展低重力精密制造、真空晶圆与"月轨金融"，发行以氦-3为锚的清算货币"潮汐元"'


In [13]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

静海城位于月球北半球静海（Mare Tranquillitatis）西缘，阿波罗11号着陆区以南约80公里，依托熔岩管群与月海玄武岩台地建造，地下城体深达200米，地表以穹顶港和升降井与轨道电梯相连，是一座冷寂与繁华并存。地表是永恒的黑白天穹与低重力尘暴，地下是永不熄灭的霓虹熔岩廊道；居民被称为"潮汐人"，说话慢、走路轻，习惯以敲击管壁代替敲门。整座城按地球时运行，却把"月昼"当作节日——每逢月昼开端，全城穹顶会同步降下人造暮色，人们称之为"长夜礼"的城市，其主要产业包括以氦-3开采与聚变燃料精炼为支柱，垄断地月能源贸易；水冰冶炼与氧氮分离支撑生命维持系统出口；月球轨道电梯使静海城成为地月物流中枢，同时发展低重力精密制造、真空晶圆与"月轨金融"，发行以氦-3为锚的清算货币"潮汐元"。
